# Course-thickness transition Monte Carlo

Edit the parameters below and run all cells. A turning is counted as a match when its elevation lies within the specified interval immediately below a detected thickness-range transition.

In [ ]:
import numpy as np
import pandas as pd

# Turning elevations (m)
TURNINGS = [27.25, 49.51, 67.68, 82.51, 94.62,
            104.51, 112.58, 119.17, 124.55, 128.94]

# Monte Carlo parameters
N_SIM = 2000
N_COURSES = 203
SEED = 42
MEAN_M = 0.722
SD_M = 0.235
LOW_M = 0.495
HIGH_M = 1.50

# Thickness-range transition parameters
MEAN_RATIO = 1.12
RANGE_RATIO = 1.06
STEP_RATIO = 1.18
MATCH_WINDOW_M = 1.50
THRESHOLD_MATCHES = 5


## Transition criterion

At a candidate boundary, the algorithm uses two courses before and three courses after. It requires:

1. mean(first two after) ≥ `MEAN_RATIO` × mean(two before);
2. minimum(first two after) ≥ `RANGE_RATIO` × maximum(two before);
3. first course after ≥ `STEP_RATIO` × immediately preceding course;
4. the first three courses after satisfy first > second > third.

In [ ]:
def draw_truncated_normal(rng, n):
    values = np.empty(n, dtype=float)
    filled = 0
    while filled < n:
        batch = rng.normal(MEAN_M, SD_M, size=max(64, 2 * (n-filled)))
        batch = batch[(batch >= LOW_M) & (batch <= HIGH_M)]
        take = min(len(batch), n-filled)
        if take:
            values[filled:filled+take] = batch[:take]
            filled += take
    return values

def detect_transitions(thickness):
    h = np.asarray(thickness, dtype=float)
    cumulative = np.cumsum(h)
    elevations = []
    for i in range(2, len(h)-2):
        before = h[i-2:i]
        after2 = h[i:i+2]
        if (after2.mean() >= MEAN_RATIO * before.mean()
            and after2.min() >= RANGE_RATIO * before.max()
            and h[i] >= STEP_RATIO * h[i-1]
            and h[i] > h[i+1] > h[i+2]):
            elevations.append(cumulative[i-1])
    return np.asarray(elevations, dtype=float)

def score_turnings(transitions):
    transitions = np.asarray(transitions, dtype=float)
    if transitions.size == 0:
        return 0
    return int(sum(np.any((t >= transitions-MATCH_WINDOW_M) & (t <= transitions))
                   for t in TURNINGS))


In [ ]:
rng = np.random.default_rng(SEED)
scores = np.empty(N_SIM, dtype=int)
n_transitions = np.empty(N_SIM, dtype=int)

for s in range(N_SIM):
    h = draw_truncated_normal(rng, N_COURSES)
    transitions = detect_transitions(h)
    scores[s] = score_turnings(transitions)
    n_transitions[s] = len(transitions)

values, counts = np.unique(scores, return_counts=True)
distribution = pd.DataFrame({
    'matches': values,
    'simulations': counts,
    'frequency': counts / N_SIM
})

n_tail = int(np.sum(scores >= THRESHOLD_MATCHES))
summary = pd.DataFrame({
    'metric': ['seed','simulations','courses_per_simulation','number_of_turnings',
               'mean_detected_transitions','max_matches','simulations_at_or_above_threshold',
               'frequency_at_or_above_threshold','monte_carlo_p_plus'],
    'value': [SEED,N_SIM,N_COURSES,len(TURNINGS),n_transitions.mean(),scores.max(),
              n_tail,n_tail/N_SIM,(n_tail+1)/(N_SIM+1)]
})

display(distribution)
display(summary)


In [ ]:
raw_results = pd.DataFrame({
    'simulation': np.arange(1, N_SIM+1),
    'matches': scores,
    'detected_transitions': n_transitions
})

distribution.to_csv('random_heights_distribution.csv', index=False)
summary.to_csv('random_heights_summary.csv', index=False)
raw_results.to_csv('random_heights_all_simulations.csv', index=False)
